# Swing trading-agent — backtest (paper trading)

Trend/momentum-strategi, 3 måneders rebalanceringscyklus, globalt univers (US/Europa/Norden/EM), maks 5 positioner, 100.000 kr. simuleret kapital.

**Ingen ægte handler. Ingen Nordnet-integration.** Kun backtest på historiske data.

**Sådan bruger du denne notebook:** Menuen foroven → **Kør alle** (Runtime/Kørselstid → Kør alle celler). Det er det hele. Resultatet downloades automatisk til din telefon som `report.html` i sidste celle.

Første kørsel tager et par minutter (henter ~80 aktiers historik). Genkør senere for at opdatere med nye kurser.

## 1) Installér afhængigheder

In [ ]:
!pip install -q "yfinance==0.2.55" "pandas>=2.0" "numpy>=1.24"
print("Afhængigheder installeret.")

## 2) Projektets kildekode
Skriver alle moduler til disk i Colab-miljøet (skjult detalje — du behøver ikke røre disse celler).

In [ ]:
%%writefile universe.py
"""
Aktieunivers for swing trading-agenten.

Tickere er i Yahoo Finance-format (bruges af yfinance). Listen er en kurateret
udvælgelse af likvide large/mid-cap aktier på tværs af fire regioner. Formålet
er at give momentum-strategien et bredt, men håndterbart univers at rangere.

Bemærk: Dette er IKKE det samme som "alle aktier man kan handle på Nordnet" —
det er et repræsentativt udsnit. Nordnet-udbuddet er bredere (og varierer med
kontotype/marked), men de fleste af nedenstående kan handles direkte via
Nordnet på deres respektive hjemmemarkeder, evt. som ADR/depotbeviser for
enkelte EM-navne.
"""

US = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA", "AVGO", "ORCL",
    "CRM", "ADBE", "NFLX", "COST", "WMT", "HD", "PG", "KO", "PEP", "JNJ",
    "UNH", "PFE", "JPM", "BAC", "V", "MA", "XOM", "CVX", "DIS", "INTC", "CSCO",
]

EUROPE = [
    "ASML.AS",   # ASML
    "SAP.DE",    # SAP
    "MC.PA",     # LVMH
    "OR.PA",     # L'Oréal
    "TTE.PA",    # TotalEnergies
    "AIR.PA",    # Airbus
    "SAN.PA",    # Sanofi
    "SIE.DE",    # Siemens
    "ALV.DE",    # Allianz
    "BAS.DE",    # BASF
    "NOVN.SW",   # Novartis
    "NESN.SW",   # Nestlé
    "ROG.SW",    # Roche
    "AZN.L",     # AstraZeneca
    "SHEL.L",    # Shell
    "HSBA.L",    # HSBC
    "ULVR.L",    # Unilever
    "IBE.MC",    # Iberdrola
]

NORDIC = [
    "NOVO-B.CO",   # Novo Nordisk
    "MAERSK-B.CO", # Maersk
    "VWS.CO",      # Vestas
    "ORSTED.CO",   # Ørsted
    "DSV.CO",      # DSV
    "GMAB.CO",     # Genmab
    "COLO-B.CO",   # Coloplast
    "ERIC-B.ST",   # Ericsson
    "VOLV-B.ST",   # Volvo
    "ATCO-A.ST",   # Atlas Copco
    "INVE-B.ST",   # Investor AB
    "HM-B.ST",     # H&M
    "SAND.ST",     # Sandvik
    "EQNR.OL",     # Equinor
    "DNB.OL",      # DNB
    "TEL.OL",      # Telenor
    "NOKIA.HE",    # Nokia
    "SAMPO.HE",    # Sampo
]

EMERGING_MARKETS = [
    "TSM",    # Taiwan Semiconductor (ADR)
    "BABA",   # Alibaba (ADR)
    "PDD",    # PDD Holdings (ADR)
    "JD",     # JD.com (ADR)
    "TCEHY",  # Tencent (ADR)
    "INFY",   # Infosys (ADR)
    "IBN",    # ICICI Bank (ADR)
    "HDB",    # HDFC Bank (ADR)
    "MELI",   # MercadoLibre
    "VALE",   # Vale
    "ITUB",   # Itaú Unibanco (ADR)
    "PBR",    # Petrobras (ADR)
    "AMX",    # América Móvil (ADR)
]

# Benchmark: Jacobi har allerede iShares MSCI ACWI UCITS ETF (IE00B6R52259) i
# ratepensionen. Det US-listede søster-produkt "ACWI" (iShares MSCI ACWI ETF)
# bruges her som benchmark, da det er tilgængeligt via yfinance og
# repræsenterer samme globale aktieeksponering.
BENCHMARK = "ACWI"

def full_universe():
    """Return the full ticker list (all regions), de-duplicated, order preserved."""
    seen = set()
    out = []
    for group in (US, EUROPE, NORDIC, EMERGING_MARKETS):
        for t in group:
            if t not in seen:
                seen.add(t)
                out.append(t)
    return out

def region_of(ticker):
    if ticker in US:
        return "US"
    if ticker in EUROPE:
        return "Europa"
    if ticker in NORDIC:
        return "Norden"
    if ticker in EMERGING_MARKETS:
        return "Emerging Markets"
    return "Ukendt"


In [ ]:
%%writefile config.py
"""
Strategi- og backtest-parametre. Alt her er bevidst samlet ét sted, så du kan
tune og genkøre uden at rode i strategilogikken.
"""

from dataclasses import dataclass


@dataclass
class Config:
    # --- Backtest-periode ---
    start_date: str = "2015-01-01"
    end_date: str = None  # None = i dag

    # --- Portefølje ---
    starting_capital: float = 100_000.0   # simuleret kapital (paper trading)
    max_positions: int = 5
    weighting: str = "equal"              # "equal" eller "inverse_vol"

    # --- Rebalancering (3 måneders cyklus) ---
    rebalance_every_days: int = 63        # ~1 handelskvartal

    # --- Trend/momentum-signal ---
    trend_sma_window: int = 200           # trendfilter: kurs > SMA(200)
    momentum_lookback_days: int = 126     # ~6 mdr. samlet lookback
    momentum_skip_days: int = 21          # spring seneste ~1 mdr. over
    # Momentum-score = afkast fra (t - lookback) til (t - skip).
    # "Skip-month"-konventionen er standard i akademisk momentumforskning
    # (Jegadeesh & Titman) og undgår kortsigtet reversal-støj.

    # --- Risikostyring mellem rebalanceringer ---
    stop_loss_pct: float = 0.15           # luk position ved -15% fra indgang
    risk_check_every_days: int = 5        # tjek stop-loss ugentligt
    reinvest_after_stop: bool = False     # hold kontant til næste rebalancering

    # --- Omkostninger (approksimeret — bekræft reelle Nordnet-satser) ---
    cost_pct: float = 0.0010              # 0,10% pr. handel (køb/salg hver for sig)
    min_fee_local: float = 29.0           # simpel min.-kurtage, lokal valuta-enhed

    # --- Øvrigt ---
    min_history_days: int = 220           # aktien skal have mindst denne historik
                                            # for at være valgbar (SMA200 + margin)

    def __post_init__(self):
        if self.max_positions < 1:
            raise ValueError("max_positions skal være >= 1")
        if not (0 < self.cost_pct < 0.05):
            raise ValueError("cost_pct virker urealistisk")


In [ ]:
%%writefile data_fetch.py
"""
Henter og cacher historiske daglige kurser via yfinance.

Kør dette script direkte for at fylde cachen op, eller importér
`load_prices()` fra andre scripts.

VIGTIGT: Dette script kræver almindelig internetadgang til Yahoo Finance.
Det er IKKE testet i det sandbox-miljø hvor agenten oprindeligt blev bygget
(den havde kun adgang til pypi/npm/github) og er heller IKKE testet på iOS —
kør `python data_fetch.py --check` først for at bekræfte at det virker i dit
miljø, før du kaster hele universet på det. Se README.md, afsnittet
"Kør fra iPhone", hvis yfinance ikke vil installere/køre.

Cache-format: almindelig CSV (ikke parquet) — undgår pyarrow, som er
tung/besværlig at få installeret i mange mobile Python-apps.
"""

import os
import sys
import time
import pandas as pd

from universe import full_universe, BENCHMARK
from config import Config

CACHE_DIR = os.path.join(os.path.dirname(__file__), "cache")


def _cache_path(ticker: str) -> str:
    safe = ticker.replace("/", "_")
    return os.path.join(CACHE_DIR, f"{safe}.csv")


def fetch_ticker(ticker: str, start: str, end: str, force: bool = False) -> pd.DataFrame:
    """Fetch (or load cached) daily OHLCV for one ticker. Returns adjusted close
    as 'AdjClose' plus raw OHLCV. Empty DataFrame on failure."""
    try:
        import yfinance as yf
    except ImportError as e:
        raise ImportError(
            "yfinance kunne ikke importeres. Kør 'pip install -r requirements.txt' "
            "(pinner yfinance==0.2.55, som kun kræver rene Python-pakker). Hvis "
            "'pip install' selv fejlede på en anden pakke, så virker den formentlig "
            "ikke på din platform uden kompiler — se README.md 'Kør fra iPhone'."
        ) from e

    path = _cache_path(ticker)
    if not force and os.path.exists(path):
        try:
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            if not df.empty:
                return df
        except Exception:
            pass

    for attempt in range(3):
        try:
            df = yf.download(
                ticker, start=start, end=end, auto_adjust=False,
                progress=False, threads=False,
            )
            if df is None or df.empty:
                raise ValueError("tom respons")
            # yfinance can return MultiIndex columns for single-ticker calls
            # depending on version; normalise to flat columns.
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df.rename(columns={"Adj Close": "AdjClose"})
            df = df[["Open", "High", "Low", "Close", "AdjClose", "Volume"]].dropna(
                subset=["AdjClose"]
            )
            os.makedirs(CACHE_DIR, exist_ok=True)
            df.to_csv(path)
            return df
        except Exception as e:
            print(f"  [{ticker}] forsøg {attempt+1}/3 fejlede: {e}", file=sys.stderr)
            time.sleep(1.5 * (attempt + 1))

    print(f"  [{ticker}] KUNNE IKKE HENTES — udelades af universet.", file=sys.stderr)
    return pd.DataFrame()


def fetch_all(cfg: Config, force: bool = False) -> dict:
    """Fetch benchmark + full universe. Returns {ticker: DataFrame}."""
    end = cfg.end_date or pd.Timestamp.today().strftime("%Y-%m-%d")
    tickers = [BENCHMARK] + full_universe()
    data = {}
    print(f"Henter kursdata for {len(tickers)} tickere ({cfg.start_date} -> {end})...")
    for i, t in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] {t}")
        df = fetch_ticker(t, cfg.start_date, end, force=force)
        if not df.empty and len(df) >= cfg.min_history_days:
            data[t] = df
        elif not df.empty:
            print(f"  [{t}] for kort historik ({len(df)} dage) — udelades.")
    missing = set(tickers) - set(data.keys())
    if missing:
        print(f"\nAdvarsel: {len(missing)} tickere kunne ikke hentes/opfylder ikke "
              f"minimumshistorik: {sorted(missing)}")
    print(f"\nFærdig. {len(data)} tickere klar (inkl. benchmark).")
    return data


def load_prices(cfg: Config, force: bool = False) -> pd.DataFrame:
    """Return a single wide DataFrame of AdjClose, columns = tickers."""
    data = fetch_all(cfg, force=force)
    if not data:
        raise RuntimeError(
            "Ingen data hentet. Tjek internetforbindelse og at yfinance er "
            "installeret (pip install -r requirements.txt)."
        )
    series = {t: df["AdjClose"] for t, df in data.items()}
    wide = pd.DataFrame(series).sort_index()
    # Forward-fill korte huller (helligdage der ikke matcher på tværs af
    # børser), men dropp rækker hvor benchmark mangler.
    wide = wide.ffill(limit=5)
    return wide


def check_environment():
    """Hurtig diagnosticering: kan vi importere yfinance og hente ÉN ticker?
    Kør: python data_fetch.py --check
    Nyttigt til at fejlsøge på telefonen før hele universet hentes."""
    print("1) Importerer yfinance...")
    try:
        import yfinance as yf
        print("   OK.")
    except Exception as e:
        print(f"   FEJL: {e}")
        print("   -> yfinance er ikke installeret korrekt. Se README.md 'Kør fra iPhone'.")
        return False

    print("2) Henter 5 dages data for AAPL (test)...")
    try:
        df = yf.download("AAPL", period="5d", progress=False, threads=False)
        if df is None or df.empty:
            print("   FEJL: tom respons — sandsynligvis blokeret/rate-limited af Yahoo Finance.")
            return False
        print(f"   OK — modtog {len(df)} rækker.")
    except Exception as e:
        print(f"   FEJL: {e}")
        print("   -> Sandsynligvis netværksproblem eller anti-bot-blokering. Se README.md.")
        return False

    print("\nMiljøet ser klar ud. Kør 'python run_backtest.py' for den fulde kørsel.")
    return True


if __name__ == "__main__":
    if "--check" in sys.argv:
        ok = check_environment()
        sys.exit(0 if ok else 1)

    cfg = Config()
    force = "--force" in sys.argv
    prices = load_prices(cfg, force=force)
    out_path = os.path.join(os.path.dirname(__file__), "cache", "_prices_wide.csv")
    prices.to_csv(out_path)
    print(f"\nGemte samlet prisdata: {out_path}  (shape={prices.shape})")


In [ ]:
%%writefile strategy.py
"""
Trend/momentum swing-signal.

Alle beregninger bruger UDELUKKENDE data til og med `as_of_date` (ingen
look-ahead). Det er en simpel, gennemsigtig regelbaseret model — ikke en
sort boks:

1. Trendfilter: aktien skal handle over sit 200-dages glidende gennemsnit.
   Fjerner aktier i langsigtet nedtrend fra kandidatlisten.
2. Momentum-score: afkast fra (t - lookback) til (t - skip), altså et
   "skip-month"-momentum (fx 6 mdr. afkast, seneste måned udeladt) — en
   standardkonstruktion i akademisk momentumforskning (Jegadeesh & Titman),
   der undgår kortsigtet mean-reversion-støj lige før rebalanceringsdatoen.

Kandidater rangeres efter momentum-score; kun aktier der består trendfilteret
er i det hele taget kandidater.
"""

import pandas as pd

from config import Config


def get_signals_as_of(prices_wide: pd.DataFrame, as_of_date, cfg: Config) -> pd.DataFrame:
    """Compute trend + momentum signals for every ticker, using only data up to
    and including as_of_date. Returns a DataFrame indexed by ticker with columns
    [last_price, sma, trend_ok, momentum]. Tickers without enough history are
    dropped (NaN)."""
    hist = prices_wide.loc[:as_of_date]
    n = len(hist)
    needed = max(cfg.trend_sma_window, cfg.momentum_lookback_days) + 1
    if n < needed:
        return pd.DataFrame(columns=["last_price", "sma", "trend_ok", "momentum"])

    sma = hist.rolling(cfg.trend_sma_window, min_periods=cfg.trend_sma_window).mean().iloc[-1]
    last = hist.iloc[-1]
    trend_ok = last > sma

    skip_idx = -1 - cfg.momentum_skip_days
    look_idx = -1 - cfg.momentum_lookback_days
    if abs(look_idx) > n or abs(skip_idx) > n:
        return pd.DataFrame(columns=["last_price", "sma", "trend_ok", "momentum"])

    p_skip = hist.iloc[skip_idx]
    p_look = hist.iloc[look_idx]
    momentum = (p_skip / p_look) - 1.0

    out = pd.DataFrame({
        "last_price": last,
        "sma": sma,
        "trend_ok": trend_ok,
        "momentum": momentum,
    })
    return out


def rank_candidates(prices_wide: pd.DataFrame, as_of_date, cfg: Config,
                     exclude=None) -> pd.DataFrame:
    """Return candidates passing the trend filter, sorted by momentum desc.
    `exclude` is a set of tickers to never consider (e.g. the benchmark)."""
    exclude = exclude or set()
    sig = get_signals_as_of(prices_wide, as_of_date, cfg)
    if sig.empty:
        return sig
    sig = sig.dropna()
    sig = sig[sig["trend_ok"]]
    sig = sig[~sig.index.isin(exclude)]
    sig = sig.sort_values("momentum", ascending=False)
    return sig


def target_weights(candidates: pd.DataFrame, prices_wide: pd.DataFrame,
                    as_of_date, cfg: Config) -> pd.Series:
    """Pick the top max_positions candidates and assign portfolio weights.
    Returns a Series {ticker: weight}, weights summing to <= 1.0 (remainder
    is cash if fewer than max_positions candidates qualify)."""
    top = candidates.head(cfg.max_positions)
    if top.empty:
        return pd.Series(dtype=float)

    if cfg.weighting == "equal":
        w = pd.Series(1.0 / cfg.max_positions, index=top.index)
    elif cfg.weighting == "inverse_vol":
        hist = prices_wide.loc[:as_of_date, top.index].tail(63)
        daily_ret = hist.pct_change().dropna()
        vol = daily_ret.std()
        vol = vol.replace(0, vol.mean())
        inv_vol = 1.0 / vol
        raw = inv_vol / inv_vol.sum()
        # scale so the whole basket still uses at most max_positions "slots"
        # worth of capital (comparable exposure to equal-weight)
        w = raw * (min(len(top), cfg.max_positions) / cfg.max_positions)
    else:
        raise ValueError(f"Ukendt weighting-metode: {cfg.weighting}")

    return w


In [ ]:
%%writefile backtest.py
"""
Porteføljesimulering (backtest-motor).

Simulerer dag-for-dag med to typer beslutningspunkter:

- Rebalancerings-dage (hver ~63 handelsdage / 1 kvartal): kør trend/momentum-
  rangering, luk positioner der ikke længere er i top-N, åbn nye i top-N.
  Positioner der beholdes fra forrige cyklus rebalanceres IKKE i vægt (mindre
  unødig omsætning/omkostning — realistisk swing trading-adfærd).
- Risikotjek-dage (hver ~5 handelsdage / ugentligt): tjek stop-loss på åbne
  positioner. Ved brud sælges positionen, og provenuet forbliver kontant til
  næste rebalancering (konfigurerbart via cfg.reinvest_after_stop).

Al handel eksekveres til dagens slutkurs (close) på beslutningsdagen selv —
en almindelig forsimpling i denne slags backtests. En mere konservativ
variant ville eksekvere til NÆSTE dags åbningskurs; det er en oplagt
udvidelse hvis du vil gøre simuleringen strengere.

VIGTIG BEGRÆNSNING — valuta: aktierne i universet handles i flere valutaer
(USD, EUR, DKK, SEK, NOK, CHF, GBP m.fl.). Denne motor regner porteføljens
værdi som en vægtet sum af hver akties LOKALE procentafkast — den konverterer
IKKE til en fælles valuta og medregner IKKE valutakursudsving eller Nordnets
vekslingsgebyr (typisk ~0,25-0,5% pr. handel i fremmed valuta). Til reel
handel skal FX-eksponering lægges ind som en ekstra omkostnings- og
risikofaktor.
"""

import pandas as pd
import numpy as np

from config import Config
from strategy import rank_candidates, target_weights
from universe import BENCHMARK


def _trade_cost(notional: float, cfg: Config) -> float:
    return max(abs(notional) * cfg.cost_pct, cfg.min_fee_local)


def run_backtest(prices_wide: pd.DataFrame, cfg: Config):
    """Run the swing strategy over the full price history.

    Returns a dict with:
      equity_curve: pd.Series (date -> total portfolio value)
      benchmark_curve: pd.Series (date -> benchmark value, same starting capital)
      trade_log: list of dicts (closed trades)
      holdings_log: list of dicts (rebalance snapshots, for the report)
    """
    dates = prices_wide.index
    universe_cols = [c for c in prices_wide.columns if c != BENCHMARK]

    needed = max(cfg.trend_sma_window, cfg.momentum_lookback_days) + 1
    if len(dates) <= needed:
        raise RuntimeError("Ikke nok historik til at starte backtesten.")

    start_i = needed
    cash = cfg.starting_capital
    positions = {}  # ticker -> {"shares": float, "entry_price": float, "entry_date": Timestamp}
    equity_curve = {}
    trade_log = []
    holdings_log = []

    last_rebalance_i = None
    last_risk_check_i = None

    def portfolio_value(i):
        val = cash
        px = prices_wide.iloc[i]
        for t, pos in positions.items():
            price = px.get(t, np.nan)
            if pd.notna(price):
                val += pos["shares"] * price
        return val

    def close_position(t, i, reason):
        nonlocal cash
        price = prices_wide.iloc[i][t]
        pos = positions.pop(t)
        notional = pos["shares"] * price
        fee = _trade_cost(notional, cfg)
        cash += notional - fee
        holding_days = (dates[i] - pos["entry_date"]).days
        ret_pct = (price / pos["entry_price"]) - 1.0
        trade_log.append({
            "ticker": t,
            "entry_date": pos["entry_date"],
            "exit_date": dates[i],
            "entry_price": pos["entry_price"],
            "exit_price": price,
            "return_pct": ret_pct,
            "holding_days": holding_days,
            "reason": reason,
        })

    def open_position(t, i, weight, total_value):
        nonlocal cash
        price = prices_wide.iloc[i][t]
        if pd.isna(price) or price <= 0:
            return
        target_notional = total_value * weight
        fee_estimate = _trade_cost(target_notional, cfg)
        spend = max(target_notional - fee_estimate, 0)
        shares = spend / price
        actual_notional = shares * price
        fee = _trade_cost(actual_notional, cfg)
        total_cost = actual_notional + fee
        if total_cost > cash:
            shares = cash / (price * (1 + cfg.cost_pct))
            actual_notional = shares * price
            fee = _trade_cost(actual_notional, cfg)
            total_cost = actual_notional + fee
        if shares <= 0:
            return
        cash -= total_cost
        positions[t] = {"shares": shares, "entry_price": price, "entry_date": dates[i]}

    for i in range(start_i, len(dates)):
        date = dates[i]

        # --- risk check (stop-loss) ---
        is_risk_day = (last_risk_check_i is None) or (i - last_risk_check_i >= cfg.risk_check_every_days)
        if is_risk_day:
            last_risk_check_i = i
            for t in list(positions.keys()):
                price = prices_wide.iloc[i].get(t, np.nan)
                if pd.isna(price):
                    continue
                entry = positions[t]["entry_price"]
                if (price / entry - 1.0) <= -cfg.stop_loss_pct:
                    close_position(t, i, reason="stop_loss")

        # --- rebalance ---
        is_rebal_day = (last_rebalance_i is None) or (i - last_rebalance_i >= cfg.rebalance_every_days)
        if is_rebal_day:
            last_rebalance_i = i
            candidates = rank_candidates(prices_wide[universe_cols], date, cfg, exclude=set())
            weights = target_weights(candidates, prices_wide, date, cfg)
            target_set = set(weights.index)

            # close positions no longer in target
            for t in list(positions.keys()):
                if t not in target_set:
                    close_position(t, i, reason="rebalance_rotate")

            # open new positions from target not currently held
            total_value = portfolio_value(i)
            new_names = [t for t in weights.index if t not in positions]
            for t in new_names:
                open_position(t, i, weights[t], total_value)

            holdings_log.append({
                "date": date,
                "holdings": list(positions.keys()),
                "candidates_considered": len(candidates),
                "portfolio_value": portfolio_value(i),
            })

        equity_curve[date] = portfolio_value(i)

    equity_curve = pd.Series(equity_curve).sort_index()

    # Benchmark: buy-and-hold from the same start date, same starting capital,
    # no costs (a clean reference line).
    bench_px = prices_wide[BENCHMARK].loc[equity_curve.index[0]:equity_curve.index[-1]]
    bench_curve = cfg.starting_capital * (bench_px / bench_px.iloc[0])

    return {
        "equity_curve": equity_curve,
        "benchmark_curve": bench_curve,
        "trade_log": trade_log,
        "holdings_log": holdings_log,
        "final_positions": positions,
        "final_cash": cash,
    }


In [ ]:
%%writefile metrics.py
"""
Performance-nøgletal ud fra en equity-kurve og en handelslog.

Antagelser (vigtige at kende når tallene skal fortolkes):
- Sharpe-ratio antager 0% risikofri rente (forsimpling).
- CAGR beregnes ud fra kalenderdage mellem første og sidste dato.
- Alle afkast er FØR skat.
"""

import numpy as np
import pandas as pd


def cagr(equity: pd.Series) -> float:
    if len(equity) < 2:
        return float("nan")
    years = (equity.index[-1] - equity.index[0]).days / 365.25
    if years <= 0:
        return float("nan")
    total_return = equity.iloc[-1] / equity.iloc[0]
    return total_return ** (1 / years) - 1


def max_drawdown(equity: pd.Series):
    running_max = equity.cummax()
    dd = equity / running_max - 1.0
    trough_date = dd.idxmin()
    max_dd = dd.min()
    peak_date = equity.loc[:trough_date].idxmax()
    return max_dd, peak_date, trough_date


def sharpe_ratio(equity: pd.Series, periods_per_year: int = 252, rf: float = 0.0) -> float:
    daily_ret = equity.pct_change().dropna()
    if daily_ret.std() == 0 or len(daily_ret) < 2:
        return float("nan")
    excess = daily_ret - rf / periods_per_year
    return np.sqrt(periods_per_year) * excess.mean() / daily_ret.std()


def quarterly_returns(equity: pd.Series) -> pd.DataFrame:
    q = equity.resample("QE").last()
    q_ret = q.pct_change()
    df = pd.DataFrame({"portfolio_value": q, "return": q_ret})
    df = df.dropna(subset=["return"])
    return df


def trade_stats(trade_log: list) -> dict:
    if not trade_log:
        return {
            "n_trades": 0, "win_rate": float("nan"),
            "avg_return_pct": float("nan"), "avg_holding_days": float("nan"),
            "best_trade_pct": float("nan"), "worst_trade_pct": float("nan"),
        }
    df = pd.DataFrame(trade_log)
    wins = (df["return_pct"] > 0).sum()
    return {
        "n_trades": len(df),
        "win_rate": wins / len(df),
        "avg_return_pct": df["return_pct"].mean(),
        "avg_holding_days": df["holding_days"].mean(),
        "best_trade_pct": df["return_pct"].max(),
        "worst_trade_pct": df["return_pct"].min(),
    }


def summary(equity: pd.Series, benchmark: pd.Series, trade_log: list) -> dict:
    dd, peak_dt, trough_dt = max_drawdown(equity)
    bench_dd, _, _ = max_drawdown(benchmark)

    return {
        "start_date": equity.index[0],
        "end_date": equity.index[-1],
        "start_value": equity.iloc[0],
        "end_value": equity.iloc[-1],
        "total_return": equity.iloc[-1] / equity.iloc[0] - 1,
        "cagr": cagr(equity),
        "max_drawdown": dd,
        "max_drawdown_peak": peak_dt,
        "max_drawdown_trough": trough_dt,
        "sharpe": sharpe_ratio(equity),
        "bench_total_return": benchmark.iloc[-1] / benchmark.iloc[0] - 1,
        "bench_cagr": cagr(benchmark),
        "bench_max_drawdown": bench_dd,
        "bench_sharpe": sharpe_ratio(benchmark),
        **{f"trade_{k}": v for k, v in trade_stats(trade_log).items()},
    }


In [ ]:
%%writefile report.py
"""
Genererer en selvstændig HTML-rapport med grafer og nøgletal.

Ingen matplotlib — graferne er håndbyggede inline SVG'er i ren Python
(ingen ekstra afhængighed, kører overalt, også i letvægts iOS/Android
Python-apps). Farvevalg følger Anthropics dataviz-retningslinjer
(kategorisk rækkefølge, statusfarver forbeholdt op/ned, lav-kontrast
gitterlinjer, ingen dobbelt y-akse).
"""

import os
import html as _html

import pandas as pd

from config import Config
from universe import region_of

# --- Farvepalet (fra dataviz-skillet, lys tilstand) ---
SURFACE = "#fcfcfb"
PAGE = "#f9f9f7"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"
SERIES_BLUE = "#2a78d6"     # kategorisk slot 1 — strategi
SERIES_ORANGE = "#eb6834"   # kategorisk slot 2 — benchmark
SERIES_RED = "#e34948"      # kategorisk slot 8 — drawdown
STATUS_GOOD = "#0ca30c"
STATUS_CRITICAL = "#d03b3b"


# ---------------------------------------------------------------------
# SVG-hjælpefunktioner (ingen matplotlib, ingen eksterne afhængigheder)
# ---------------------------------------------------------------------

def _scale(v, vmin, vmax, out_min, out_max):
    if vmax == vmin:
        return (out_min + out_max) / 2
    return out_min + (v - vmin) / (vmax - vmin) * (out_max - out_min)


def _nice_ticks(vmin, vmax, n=5):
    if vmin == vmax:
        vmin -= 1
        vmax += 1
    span = vmax - vmin
    step_raw = span / max(n - 1, 1)
    mag = 10 ** (len(str(int(step_raw))) - 1) if step_raw >= 1 else 1
    for mult in (1, 2, 2.5, 5, 10):
        step = mag * mult
        if step >= step_raw:
            break
    start = (vmin // step) * step
    ticks = []
    t = start
    while t <= vmax + step:
        ticks.append(t)
        t += step
    return ticks


def line_chart_svg(series_dict: dict, colors: dict, title: str,
                    width=720, height=280, y_fmt=lambda v: f"{v:.0f}") -> str:
    """series_dict: {name: pd.Series} sharing a common (already-aligned) index."""
    pad_l, pad_r, pad_t, pad_b = 52, 16, 40, 30
    plot_w = width - pad_l - pad_r
    plot_h = height - pad_t - pad_b

    idx = next(iter(series_dict.values())).index
    n = len(idx)
    all_vals = pd.concat(series_dict.values())
    vmin, vmax = float(all_vals.min()), float(all_vals.max())
    vspan = (vmax - vmin) or 1
    vmin -= vspan * 0.06
    vmax += vspan * 0.06

    def x_at(i):
        return pad_l + (i / max(n - 1, 1)) * plot_w

    def y_at(v):
        return pad_t + plot_h - _scale(v, vmin, vmax, 0, plot_h)

    ticks = _nice_ticks(vmin, vmax, 5)
    grid_svg = []
    for t in ticks:
        if t < vmin or t > vmax:
            continue
        y = y_at(t)
        grid_svg.append(f'<line x1="{pad_l}" y1="{y:.1f}" x2="{width-pad_r}" y2="{y:.1f}" '
                         f'stroke="{GRIDLINE}" stroke-width="1"/>')
        grid_svg.append(f'<text x="{pad_l-8}" y="{y+3:.1f}" text-anchor="end" '
                         f'font-size="10" fill="{INK_MUTED}">{y_fmt(t)}</text>')

    # x-axis year labels (sparse)
    n_labels = min(6, n)
    x_labels = []
    if n > 1:
        step = max(n // n_labels, 1)
        for i in range(0, n, step):
            x_labels.append(
                f'<text x="{x_at(i):.1f}" y="{height-pad_b+18}" text-anchor="middle" '
                f'font-size="10" fill="{INK_MUTED}">{idx[i].strftime("%Y")}</text>'
            )

    lines_svg = []
    legend_svg = []
    for j, (name, s) in enumerate(series_dict.items()):
        pts = " ".join(f"{x_at(i):.1f},{y_at(v):.1f}" for i, v in enumerate(s.values))
        color = colors[name]
        lines_svg.append(f'<polyline points="{pts}" fill="none" stroke="{color}" '
                          f'stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>')
        lx = pad_l + j * 150
        legend_svg.append(f'<line x1="{lx}" y1="14" x2="{lx+16}" y2="14" stroke="{color}" stroke-width="3"/>')
        legend_svg.append(f'<text x="{lx+22}" y="18" font-size="11" fill="{INK_SECONDARY}">{_html.escape(name)}</text>')

    baseline_y = pad_t + plot_h
    svg = f"""<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{_html.escape(title)}">
      <rect x="0" y="0" width="{width}" height="{height}" fill="{SURFACE}"/>
      <text x="{pad_l}" y="20" font-size="13" fill="{INK_PRIMARY}" font-weight="600">{_html.escape(title)}</text>
      {''.join(legend_svg)}
      {''.join(grid_svg)}
      <line x1="{pad_l}" y1="{baseline_y}" x2="{width-pad_r}" y2="{baseline_y}" stroke="{BASELINE}" stroke-width="1"/>
      {''.join(lines_svg)}
      {''.join(x_labels)}
    </svg>"""
    return svg


def area_chart_svg(series: pd.Series, color: str, title: str,
                    width=720, height=200, y_fmt=lambda v: f"{v:.0f}%") -> str:
    pad_l, pad_r, pad_t, pad_b = 52, 16, 40, 26
    plot_w = width - pad_l - pad_r
    plot_h = height - pad_t - pad_b
    n = len(series)
    vmin, vmax = float(series.min()), 0.0
    if vmin == vmax:
        vmin = -1.0
    vspan = (vmax - vmin) or 1

    def x_at(i):
        return pad_l + (i / max(n - 1, 1)) * plot_w

    def y_at(v):
        return pad_t + plot_h - _scale(v, vmin, vmax, 0, plot_h)

    ticks = _nice_ticks(vmin, vmax, 4)
    grid_svg = []
    for t in ticks:
        if t < vmin or t > vmax:
            continue
        y = y_at(t)
        grid_svg.append(f'<line x1="{pad_l}" y1="{y:.1f}" x2="{width-pad_r}" y2="{y:.1f}" '
                         f'stroke="{GRIDLINE}" stroke-width="1"/>')
        grid_svg.append(f'<text x="{pad_l-8}" y="{y+3:.1f}" text-anchor="end" '
                         f'font-size="10" fill="{INK_MUTED}">{y_fmt(t)}</text>')

    pts = [(x_at(i), y_at(v)) for i, v in enumerate(series.values)]
    poly_pts = " ".join(f"{x:.1f},{y:.1f}" for x, y in pts)
    zero_y = y_at(0)
    area_pts = f"{pad_l:.1f},{zero_y:.1f} " + poly_pts + f" {x_at(n-1):.1f},{zero_y:.1f}"

    n_labels = min(6, n)
    x_labels = []
    if n > 1:
        step = max(n // n_labels, 1)
        for i in range(0, n, step):
            x_labels.append(
                f'<text x="{x_at(i):.1f}" y="{height-pad_b+18}" text-anchor="middle" '
                f'font-size="10" fill="{INK_MUTED}">{series.index[i].strftime("%Y")}</text>'
            )

    svg = f"""<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{_html.escape(title)}">
      <rect x="0" y="0" width="{width}" height="{height}" fill="{SURFACE}"/>
      <text x="{pad_l}" y="20" font-size="13" fill="{INK_PRIMARY}" font-weight="600">{_html.escape(title)}</text>
      {''.join(grid_svg)}
      <polygon points="{area_pts}" fill="{color}" opacity="0.22" stroke="none"/>
      <polyline points="{poly_pts}" fill="none" stroke="{color}" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round"/>
      <line x1="{pad_l}" y1="{zero_y:.1f}" x2="{width-pad_r}" y2="{zero_y:.1f}" stroke="{BASELINE}" stroke-width="1"/>
      {''.join(x_labels)}
    </svg>"""
    return svg


def bar_chart_svg(q_df: pd.DataFrame, title: str, width=720, height=260) -> str:
    pad_l, pad_r, pad_t, pad_b = 52, 16, 40, 54
    plot_w = width - pad_l - pad_r
    plot_h = height - pad_t - pad_b
    n = len(q_df)
    vals = (q_df["return"] * 100)
    vmin, vmax = float(vals.min()), float(vals.max())
    vmin = min(vmin, 0)
    vmax = max(vmax, 0)
    vspan = (vmax - vmin) or 1
    vmin -= vspan * 0.12
    vmax += vspan * 0.12

    def y_at(v):
        return pad_t + plot_h - _scale(v, vmin, vmax, 0, plot_h)

    zero_y = y_at(0)
    bw = plot_w / max(n, 1) * 0.6
    gap = plot_w / max(n, 1)

    ticks = _nice_ticks(vmin, vmax, 4)
    grid_svg = []
    for t in ticks:
        if t < vmin or t > vmax:
            continue
        y = y_at(t)
        grid_svg.append(f'<line x1="{pad_l}" y1="{y:.1f}" x2="{width-pad_r}" y2="{y:.1f}" '
                         f'stroke="{GRIDLINE}" stroke-width="1"/>')
        grid_svg.append(f'<text x="{pad_l-8}" y="{y+3:.1f}" text-anchor="end" '
                         f'font-size="10" fill="{INK_MUTED}">{t:.0f}%</text>')

    bars_svg = []
    for i, (dt, row) in enumerate(q_df.iterrows()):
        v = row["return"] * 100
        color = STATUS_GOOD if v >= 0 else STATUS_CRITICAL
        cx = pad_l + gap * i + gap / 2
        y_top = min(zero_y, y_at(v))
        h = abs(y_at(v) - zero_y)
        bars_svg.append(f'<rect x="{cx-bw/2:.1f}" y="{y_top:.1f}" width="{bw:.1f}" height="{max(h,1):.1f}" '
                         f'fill="{color}" rx="2"/>')
        label_y = y_at(v) - 5 if v >= 0 else y_at(v) + 12
        bars_svg.append(f'<text x="{cx:.1f}" y="{label_y:.1f}" text-anchor="middle" font-size="8" '
                         f'fill="{INK_SECONDARY}">{v:+.1f}%</text>')
        bars_svg.append(f'<text x="{cx:.1f}" y="{height-pad_b+16}" text-anchor="middle" font-size="8" '
                         f'fill="{INK_MUTED}" transform="rotate(-45 {cx:.1f} {height-pad_b+16})">'
                         f'{dt.year} K{dt.quarter}</text>')

    svg = f"""<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{_html.escape(title)}">
      <rect x="0" y="0" width="{width}" height="{height}" fill="{SURFACE}"/>
      <text x="{pad_l}" y="20" font-size="13" fill="{INK_PRIMARY}" font-weight="600">{_html.escape(title)}</text>
      {''.join(grid_svg)}
      <line x1="{pad_l}" y1="{zero_y:.1f}" x2="{width-pad_r}" y2="{zero_y:.1f}" stroke="{BASELINE}" stroke-width="1"/>
      {''.join(bars_svg)}
    </svg>"""
    return svg


def _weekly(series: pd.Series) -> pd.Series:
    return series.resample("W").last().dropna()


def build_equity_svg(equity: pd.Series, benchmark: pd.Series) -> str:
    idx_equity = _weekly(equity / equity.iloc[0] * 100)
    idx_bench = _weekly(benchmark / benchmark.iloc[0] * 100)
    common = idx_equity.index.union(idx_bench.index)
    idx_equity = idx_equity.reindex(common).ffill().bfill()
    idx_bench = idx_bench.reindex(common).ffill().bfill()
    return line_chart_svg(
        {"Swing-agent": idx_equity, "Benchmark (ACWI)": idx_bench},
        {"Swing-agent": SERIES_BLUE, "Benchmark (ACWI)": SERIES_ORANGE},
        "Porteføljeværdi vs. benchmark (indekseret, start = 100)",
        y_fmt=lambda v: f"{v:.0f}",
    )


def build_drawdown_svg(equity: pd.Series) -> str:
    running_max = equity.cummax()
    dd = _weekly((equity / running_max - 1.0) * 100)
    return area_chart_svg(dd, SERIES_RED, "Drawdown fra seneste top")


def build_quarterly_svg(q_df: pd.DataFrame) -> str:
    return bar_chart_svg(q_df, "Afkast pr. 3-måneders cyklus")


# ---------------------------------------------------------------------
# HTML-rapport
# ---------------------------------------------------------------------

def _fmt_pct(x):
    return "–" if pd.isna(x) else f"{x*100:+.1f}%"


def _fmt_num(x, digits=2):
    return "–" if pd.isna(x) else f"{x:.{digits}f}"


def build_html(cfg: Config, summ: dict, trade_log: list, universe_size: int,
                missing_tickers: list, current_candidates: pd.DataFrame,
                equity_svg: str, dd_svg: str, q_svg: str) -> str:

    stat_tiles = [
        ("Samlet afkast", _fmt_pct(summ["total_return"]), _fmt_pct(summ["bench_total_return"])),
        ("CAGR (årligt)", _fmt_pct(summ["cagr"]), _fmt_pct(summ["bench_cagr"])),
        ("Max drawdown", _fmt_pct(summ["max_drawdown"]), _fmt_pct(summ["bench_max_drawdown"])),
        ("Sharpe-ratio", _fmt_num(summ["sharpe"]), _fmt_num(summ["bench_sharpe"])),
    ]
    tiles_html = "".join(f"""
      <div class="tile">
        <div class="tile-label">{label}</div>
        <div class="tile-value">{val}</div>
        <div class="tile-bench">Benchmark: {bench}</div>
      </div>""" for label, val, bench in stat_tiles)

    trade_rows = "".join(f"""
      <tr>
        <td>{_html.escape(t['ticker'])}</td>
        <td>{_html.escape(region_of(t['ticker']))}</td>
        <td>{t['entry_date'].strftime('%Y-%m-%d')}</td>
        <td>{t['exit_date'].strftime('%Y-%m-%d')}</td>
        <td>{t['holding_days']}</td>
        <td class="{'pos' if t['return_pct']>=0 else 'neg'}">{_fmt_pct(t['return_pct'])}</td>
        <td>{'Stop-loss' if t['reason']=='stop_loss' else 'Rotation'}</td>
      </tr>""" for t in sorted(trade_log, key=lambda x: x["exit_date"], reverse=True)[:30])

    cand_rows = "".join(f"""
      <tr>
        <td>{_html.escape(tk)}</td>
        <td>{_html.escape(region_of(tk))}</td>
        <td>{_fmt_pct(row['momentum'])}</td>
        <td>{row['last_price']:.2f}</td>
      </tr>""" for tk, row in current_candidates.head(10).iterrows())

    missing_html = ""
    if missing_tickers:
        missing_html = f"""
        <p class="caveat">{len(missing_tickers)} tickere kunne ikke hentes og indgår ikke i
        denne kørsel: {_html.escape(", ".join(missing_tickers))}</p>"""

    n_trades = summ["trade_n_trades"]

    return f"""<!doctype html>
<html lang="da">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Swing trading-agent — backtest-rapport</title>
<style>
  body {{ background:{PAGE}; color:{INK_PRIMARY}; font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
         max-width: 800px; margin: 0 auto; padding: 24px 16px 80px; line-height:1.5; }}
  h1 {{ font-size: 1.4rem; margin-bottom: 4px; }}
  h2 {{ font-size: 1.05rem; margin-top: 36px; border-bottom: 1px solid {GRIDLINE}; padding-bottom: 6px; }}
  .subtitle {{ color:{INK_SECONDARY}; margin-top:0; font-size:0.9rem; }}
  .tiles {{ display:flex; gap:10px; flex-wrap:wrap; margin: 18px 0; }}
  .tile {{ background:{SURFACE}; border:1px solid {GRIDLINE}; border-radius:10px; padding:12px 16px; flex:1; min-width:140px; }}
  .tile-label {{ font-size:0.75rem; color:{INK_MUTED}; }}
  .tile-value {{ font-size:1.35rem; font-weight:600; margin-top:2px; }}
  .tile-bench {{ font-size:0.72rem; color:{INK_SECONDARY}; margin-top:4px; }}
  .chart {{ background:{SURFACE}; border:1px solid {GRIDLINE}; border-radius:10px; padding:8px; margin:14px 0; overflow-x:auto; }}
  .chart svg {{ width:100%; height:auto; display:block; min-width:480px; }}
  table {{ width:100%; border-collapse: collapse; font-size:0.82rem; margin-top:10px; display:block; overflow-x:auto; white-space:nowrap; }}
  th, td {{ text-align:left; padding:6px 8px; border-bottom:1px solid {GRIDLINE}; }}
  th {{ color:{INK_MUTED}; font-weight:600; font-size:0.72rem; text-transform:uppercase; }}
  td.pos {{ color:{STATUS_GOOD}; }}
  td.neg {{ color:{STATUS_CRITICAL}; }}
  .caveat {{ background:#fff8e6; border:1px solid #f0d98c; border-radius:8px; padding:10px 14px; font-size:0.82rem; color:{INK_SECONDARY}; }}
  .assumptions li {{ margin-bottom:6px; }}
  footer {{ margin-top:44px; color:{INK_MUTED}; font-size:0.75rem; }}
</style>
</head>
<body>
  <h1>Swing trading-agent — backtest-rapport</h1>
  <p class="subtitle">Trend/momentum-strategi · 3 måneders rebalanceringscyklus · maks {cfg.max_positions} positioner ·
  simuleret startkapital {cfg.starting_capital:,.0f} (paper trading, ingen ægte handler) ·
  periode {summ['start_date'].strftime('%Y-%m-%d')} – {summ['end_date'].strftime('%Y-%m-%d')}</p>

  <div class="tiles">{tiles_html}</div>

  <h2>Porteføljeudvikling</h2>
  <div class="chart">{equity_svg}</div>
  <div class="chart">{dd_svg}</div>
  <div class="chart">{q_svg}</div>

  <h2>Handelsstatistik</h2>
  <table>
    <tr><th>Antal handler</th><td>{n_trades}</td>
        <th>Andel vindere</th><td>{_fmt_pct(summ['trade_win_rate'])}</td></tr>
    <tr><th>Gns. afkast/handel</th><td>{_fmt_pct(summ['trade_avg_return_pct'])}</td>
        <th>Gns. holdeperiode</th><td>{_fmt_num(summ['trade_avg_holding_days'],0)} dage</td></tr>
    <tr><th>Bedste handel</th><td>{_fmt_pct(summ['trade_best_trade_pct'])}</td>
        <th>Værste handel</th><td>{_fmt_pct(summ['trade_worst_trade_pct'])}</td></tr>
  </table>

  <h2>Aktuelle topkandidater (seneste dato i datasættet)</h2>
  <table>
    <tr><th>Ticker</th><th>Region</th><th>Momentum (6 mdr., skip 1 mdr.)</th><th>Seneste kurs</th></tr>
    {cand_rows if cand_rows else '<tr><td colspan="4">Ingen kandidater bestod trendfilteret på seneste dato.</td></tr>'}
  </table>

  <h2>Handelslog (seneste 30)</h2>
  <table>
    <tr><th>Ticker</th><th>Region</th><th>Indgang</th><th>Udgang</th><th>Dage</th><th>Afkast</th><th>Årsag</th></tr>
    {trade_rows if trade_rows else '<tr><td colspan="7">Ingen lukkede handler i perioden.</td></tr>'}
  </table>

  <h2>Univers og datadækning</h2>
  <p>{universe_size} tickere forsøgt hentet (US, Europa, Norden, emerging markets) + benchmark.</p>
  {missing_html}

  <h2>Antagelser og begrænsninger</h2>
  <ul class="assumptions">
    <li>Simulerede/"paper" tal — INGEN ægte handler er foretaget, og resultatet er ikke en garanti for fremtidig performance.</li>
    <li>Valuta: afkast regnes i hver akties lokale valuta og vægtes sammen uden FX-konvertering. Nordnets vekslingsgebyr ved handel i fremmed valuta (typisk ~0,25–0,5%) er IKKE medregnet.</li>
    <li>Omkostninger er approksimeret ({cfg.cost_pct*100:.2f}% pr. handel, min. {cfg.min_fee_local:.0f} pr. handel) — bekræft Nordnets faktiske kurtagesatser før eventuel reel handel.</li>
    <li>Handler eksekveres til dagens slutkurs på beslutningsdagen (ingen slippage/næste-dags-udførelse modelleret).</li>
    <li>Sharpe-ratio antager 0% risikofri rente.</li>
    <li>Overlevelsesbias: universet er en fast liste af i dag likvide aktier — afnoterede/konkursramte selskaber fra perioden er ikke inkluderet.</li>
    <li>Dette er backtest på historiske data — INGEN garanti for at strategien virker fremadrettet.</li>
  </ul>

  <footer>Genereret af swing_agent-projektet. Kun til eget analysebrug — ikke finansiel rådgivning.</footer>
</body>
</html>"""


def generate_report(cfg: Config, prices_wide: pd.DataFrame, result: dict,
                     missing_tickers: list, universe_size: int, out_path: str):
    from strategy import rank_candidates
    from metrics import summary, quarterly_returns
    from universe import BENCHMARK

    equity = result["equity_curve"]
    bench = result["benchmark_curve"]
    summ = summary(equity, bench, result["trade_log"])
    q_df = quarterly_returns(equity)

    equity_svg = build_equity_svg(equity, bench)
    dd_svg = build_drawdown_svg(equity)
    q_svg = build_quarterly_svg(q_df)

    universe_cols = [c for c in prices_wide.columns if c != BENCHMARK]
    current_candidates = rank_candidates(prices_wide[universe_cols], prices_wide.index[-1], cfg)

    html_out = build_html(cfg, summ, result["trade_log"], universe_size, missing_tickers,
                           current_candidates, equity_svg, dd_svg, q_svg)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(html_out)
    return out_path, summ


In [ ]:
%%writefile run_backtest.py
#!/usr/bin/env python3
"""
Hovedscript: hent data -> kør backtest -> generér rapport.

Kør:
    python run_backtest.py            # brug cache hvis den findes
    python run_backtest.py --force    # tving frisk datahentning

Kræver almindelig internetadgang (yfinance/Yahoo Finance). Output lander i
output/report.html.
"""

import os
import sys

from config import Config
from universe import full_universe, BENCHMARK
from data_fetch import load_prices
from backtest import run_backtest
from report import generate_report


def main():
    force = "--force" in sys.argv
    cfg = Config()

    print("=== Swing trading-agent: backtest ===")
    print(f"Periode: {cfg.start_date} -> {cfg.end_date or 'i dag'}")
    print(f"Maks positioner: {cfg.max_positions} | Rebalancering: hver {cfg.rebalance_every_days} handelsdage")
    print()

    prices = load_prices(cfg, force=force)

    universe_size = len(full_universe()) + 1  # + benchmark
    missing = sorted(set(full_universe() + [BENCHMARK]) - set(prices.columns))

    print("\nKører backtest...")
    result = run_backtest(prices, cfg)

    out_path = os.path.join(os.path.dirname(__file__), "output", "report.html")
    out_path, summ = generate_report(cfg, prices, result, missing, universe_size, out_path)

    print("\n=== Resultat ===")
    print(f"Samlet afkast:  {summ['total_return']*100:+.1f}%  (benchmark: {summ['bench_total_return']*100:+.1f}%)")
    print(f"CAGR:           {summ['cagr']*100:+.1f}%  (benchmark: {summ['bench_cagr']*100:+.1f}%)")
    print(f"Max drawdown:   {summ['max_drawdown']*100:.1f}%  (benchmark: {summ['bench_max_drawdown']*100:.1f}%)")
    print(f"Sharpe:         {summ['sharpe']:.2f}  (benchmark: {summ['bench_sharpe']:.2f})")
    print(f"Antal handler:  {summ['trade_n_trades']}  |  Andel vindere: {summ['trade_win_rate']*100:.0f}%")
    print(f"\nRapport gemt: {out_path}")


if __name__ == "__main__":
    main()


## 3) Miljøtjek (hurtig test, ~5 sekunders data)

In [ ]:
!python data_fetch.py --check

## 4) Kør den fulde backtest

In [ ]:
!python run_backtest.py

## 5) Vis rapporten her i notebooken

In [ ]:
from IPython.display import HTML, display
with open('output/report.html', encoding='utf-8') as f:
    display(HTML(f.read()))

## 6) Download rapporten til din telefon

In [ ]:
from google.colab import files
files.download('output/report.html')